In [87]:
import numpy as np
import pandas as pd
import yaml
from eeg_loader import read_file, load_eeg, get_eeg_timestamps, load_stimulus, load_event
config_file_path = '/Users/khanhha/eeg-auditory-stimulus/configs/claassen_cfg.yml'  # Replace with the actual path to your config file
with open(config_file_path, 'r') as file:
    config = yaml.safe_load(file)

eeg_path = r"/Users/khanhha/GitHub-Projects/brain-waves-2.0/eeg_data_analysis/EEG_DATA/X~ X_86b44d5d-5036-4748-b666-ceff710a1e8d.EDF"
event_full_path = r"/Users/khanhha/GitHub-Projects/brain-waves-2.0/eeg_data_analysis/EEG_DATA/patient_df.csv"

In [88]:
fname, raw = load_eeg(eeg_path, config)
start_time, end_time = get_eeg_timestamps(raw)
patient_id = load_stimulus(event_full_path, start_time, end_time)

Extracting EDF parameters from /Users/khanhha/GitHub-Projects/brain-waves-2.0/eeg_data_analysis/EEG_DATA/X~ X_86b44d5d-5036-4748-b666-ceff710a1e8d.EDF...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 2052095  =      0.000 ...  4007.998 secs...
Resampling data to 512 Hz
Sampling frequency of the instance is already 512.0, returning unmodified.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 1691 samples (3.303 s)



/Users/khanhha/eeg-auditory-stimulus/data/eeg_loader.py:12: RuntimeWarning: Omitted 5 annotation(s) that were outside data range.
  raw = mne.io.read_raw_edf(eeg_path, preload=True)
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.6s


In [89]:
df = pd.read_csv(event_full_path)
df = df.drop(columns=['Unnamed: 0'])
df = df[df['patient_id'] == patient_id]
df['start_time'] = pd.to_datetime(df['start_time'], unit='s', utc=True)
df['end_time'] = pd.to_datetime(df['end_time'], unit='s', utc=True)
df.head()

,patient_id,date,trial_type,sentences,start_time,end_time,duration
168,CON001b,2024-09-17,beep,[],2024-09-17 22:38:58+00:00,2024-09-17 22:39:28+00:00,30.456603
169,CON001b,2024-09-17,lang,"[20, 9, 3, 26, 11, 19, 15, 13, 2, 1, 22, 16]",2024-09-17 22:39:30+00:00,2024-09-17 22:39:46+00:00,15.556894
170,CON001b,2024-09-17,beep,[],2024-09-17 22:39:47+00:00,2024-09-17 22:40:17+00:00,30.220895
171,CON001b,2024-09-17,lang,"[8, 31, 16, 1, 21, 24, 13, 20, 19, 23, 4, 33]",2024-09-17 22:40:19+00:00,2024-09-17 22:40:35+00:00,15.569231
172,CON001b,2024-09-17,lang,"[30, 27, 12, 22, 6, 21, 0, 2, 15, 32, 31, 10]",2024-09-17 22:40:37+00:00,2024-09-17 22:40:52+00:00,15.558622


In [90]:
def create_trial_events(df, start_time, config):
    sfreq = config['sfreq']

    # Standardize timestamp of trials (in seconds) - minus from the start time
    df['event_time_start'] = (df['start_time'] - start_time).dt.total_seconds()
    df['event_time_end'] = (df['end_time'] - start_time).dt.total_seconds()
    
    event_time_in_sec = df['event_time_start'].to_list()
    event_time_in_sec.append(df['event_time_end'].max())

    event_times_in_samples = [int(time * sfreq) for time in event_time_in_sec]
    previous_values = [0] * len(event_times_in_samples)
    event_dict = {
        'lang': 1,
        'rcmd': 2,
        'lcmd': 3,
        'beep': 4,
        'end_stim': 5
    }

    event_ids = []

    for id in df['trial_type']:
        event_ids.append(event_dict[id])
    event_ids.append(6)

    events = np.column_stack([event_times_in_samples, previous_values, event_ids])
    return events, event_dict

In [91]:
events, event_dict = create_trial_events(df, start_time, config)

array([[ 266752,       0,       4],
       [ 283136,       0,       1],
       [ 291840,       0,       4],
       [ 308224,       0,       1],
       [ 317440,       0,       1],
       [ 326144,       0,       1],
       [ 334848,       0,       1],
       [ 344064,       0,       1],
       [ 352768,       0,       1],
       [ 361472,       0,       4],
       [ 377856,       0,       1],
       [ 386560,       0,       1],
       [ 395264,       0,       1],
       [ 403968,       0,       1],
       [ 412672,       0,       1],
       [ 421376,       0,       1],
       [ 430592,       0,       1],
       [ 439296,       0,       1],
       [ 448000,       0,       1],
       [ 456704,       0,       1],
       [ 465408,       0,       3],
       [ 572928,       0,       4],
       [ 589312,       0,       1],
       [ 598016,       0,       1],
       [ 607232,       0,       3],
       [ 714240,       0,       2],
       [ 821248,       0,       1],
       [ 829952,       0,   